# 09 · Model Versioning & Model Card

**Project:** Enterprise HR AI — Day 2 Wrap-Up  
**Scope:** Lightweight manual versioning only. No MLflow, Docker, or external tooling —
those are deferred to the Enterprise Hardening phase per the project DOCX.

**Deliverables:**
1. `models/model_registry.json` — full version history of all models trained in Day 2
2. `docs/model_card.md` — standardized model card for the production model (v3)

---

In [1]:
import json
import os
from datetime import datetime

MODELS = os.path.join('..', 'models')
DOCS   = os.path.join('..', 'docs')
os.makedirs(DOCS, exist_ok=True)

print('MODELS:', os.path.abspath(MODELS))
print('DOCS  :', os.path.abspath(DOCS))

# Confirm key model files exist
for fname in ['attrition_pipeline.joblib', 'model_config.json',
              'baseline_logreg.joblib', 'scaler.joblib']:
    path = os.path.join(MODELS, fname)
    status = f'{os.path.getsize(path):,} bytes' if os.path.exists(path) else 'MISSING'
    print(f'  {fname:<40} {status}')

MODELS: C:\Users\ASUS\Desktop\enterprise_hr_ai\models
DOCS  : C:\Users\ASUS\Desktop\enterprise_hr_ai\docs
  attrition_pipeline.joblib                2,575 bytes
  model_config.json                        195 bytes
  baseline_logreg.joblib                   2,575 bytes
  scaler.joblib                            1,983 bytes


---
## Step 1 · Create `models/model_registry.json`

Three versioned entries covering every model evaluated during Day 2:
- **v1** — Baseline Logistic Regression (Step 6, unweighted, threshold=0.5)
- **v2** — XGBoost Unscaled (Step 7, default threshold=0.5, best of unweighted tree models)
- **v3** — Logistic Regression Balanced, threshold=0.40 (Step 7b — CURRENT PRODUCTION)


In [2]:
registry = {
    "project": "enterprise_hr_ai",
    "last_updated": datetime.now().strftime('%Y-%m-%d'),
    "versions": [
        {
            "version": "v1",
            "name": "baseline_logreg",
            "notebook": "06_baseline_model.ipynb",
            "algorithm": "LogisticRegression",
            "training_data": "features_scaled.csv",
            "class_weight": None,
            "threshold": 0.50,
            "metrics": {
                "precision": 0.6538,
                "recall": 0.3617,
                "f1": 0.4658,
                "roc_auc": 0.8134
            },
            "file_path": "models/baseline_logreg.joblib",
            "file_retained": True,
            "status": "superseded",
            "status_reason": "Beaten on Recall (0.3617 vs 0.7872) by v3 after class imbalance correction; retained as reference artifact only."
        },
        {
            "version": "v2",
            "name": "xgboost_unscaled",
            "notebook": "07_model_comparison.ipynb",
            "algorithm": "XGBClassifier",
            "training_data": "features_unscaled.csv",
            "class_weight": None,
            "scale_pos_weight": None,
            "threshold": 0.50,
            "metrics": {
                "precision": 0.5652,
                "recall": 0.2766,
                "f1": 0.3714,
                "roc_auc": 0.7735
            },
            "file_path": None,
            "file_retained": False,
            "status": "rejected",
            "status_reason": "Lower Recall and F1 than v1 baseline at default threshold; root cause was untreated class imbalance (no scale_pos_weight). Model object not saved to disk — it was superseded before archive step."
        },
        {
            "version": "v3",
            "name": "logreg_balanced_threshold_0.40",
            "notebook": "07b_model_comparison_balanced.ipynb",
            "algorithm": "LogisticRegression",
            "training_data": "features_scaled.csv",
            "class_weight": "balanced",
            "threshold": 0.40,
            "metrics": {
                "precision": 0.3426,
                "recall": 0.7872,
                "f1": 0.4774,
                "roc_auc": 0.8060
            },
            "confusion_matrix_at_threshold": {
                "TN": 176, "FP": 71, "FN": 10, "TP": 37
            },
            "file_path": "models/attrition_pipeline.joblib",
            "config_path": "models/model_config.json",
            "file_retained": True,
            "status": "production",
            "status_reason": "Best Recall (0.7872) and F1 (0.4774) across all 6 model variants evaluated. Class imbalance corrected with class_weight='balanced'. Threshold tuned to 0.40 to catch 37/47 test leavers while maintaining acceptable precision. SHAP analysis in Step 8 confirmed OverTime, YearsSinceLastPromotion, TotalWorkingYears, BusinessTravel, and JobLevel as top drivers — consistent with HR domain intuition."
        }
    ]
}

registry_path = os.path.join(MODELS, 'model_registry.json')
with open(registry_path, 'w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2)

print(f'Written: {os.path.abspath(registry_path)}')
print(f'Size   : {os.path.getsize(registry_path):,} bytes')
print()
print('=== models/model_registry.json (full contents) ===')
print(json.dumps(registry, indent=2))

Written: C:\Users\ASUS\Desktop\enterprise_hr_ai\models\model_registry.json
Size   : 2,640 bytes

=== models/model_registry.json (full contents) ===
{
  "project": "enterprise_hr_ai",
  "last_updated": "2026-09-01",
  "versions": [
    {
      "version": "v1",
      "name": "baseline_logreg",
      "notebook": "06_baseline_model.ipynb",
      "algorithm": "LogisticRegression",
      "training_data": "features_scaled.csv",
      "class_weight": null,
      "threshold": 0.5,
      "metrics": {
        "precision": 0.6538,
        "recall": 0.3617,
        "f1": 0.4658,
        "roc_auc": 0.8134
      },
      "file_path": "models/baseline_logreg.joblib",
      "file_retained": true,
      "status": "superseded",
      "status_reason": "Beaten on Recall (0.3617 vs 0.7872) by v3 after class imbalance correction; retained as reference artifact only."
    },
    {
      "version": "v2",
      "name": "xgboost_unscaled",
      "notebook": "07_model_comparison.ipynb",
      "algorithm": "XGBCla

---
## Step 2 · Day 2 Decision Trail

Full chronological record of every modeling decision made during Day 2.
This cell is the seed content for `docs/model_card.md`.

---

### Day 2 Attrition Modeling — Decision Trail

#### 1. Baseline Established (Notebook 06)
A Logistic Regression model was trained on `features_scaled.csv` (1,176 train / 294 test,
80/20 stratified split, `random_state=42`). No class weighting. Default threshold 0.5.
**Result:** Recall=0.3617, F1=0.4658, ROC-AUC=0.8134.
This became the benchmark every subsequent model had to beat on Recall (primary) and F1 (secondary).

#### 2. Tree Models Attempted Without Imbalance Correction (Notebook 07)
Random Forest (`n_estimators=200`) and XGBoost were evaluated on `features_unscaled.csv`
alongside an unscaled Logistic Regression re-run. All three failed to beat the baseline on
Recall and F1 at the default 0.5 threshold. Best among them: XGBoost (Recall=0.2766, F1=0.3714).
RF was the worst (Recall=0.1064). Scaling ruled out as the cause — scaled RF and XGBoost
produced identical results, confirming the real issue was the 84/16 class imbalance.

#### 3. Class Imbalance Correction Applied (Notebook 07b)
Three models retrained with explicit imbalance handling:
- `LogisticRegression(class_weight='balanced')` on scaled features
- `RandomForestClassifier(class_weight='balanced')` on unscaled features
- `XGBClassifier(scale_pos_weight=5.2)` on unscaled features

#### 4. Honest Re-Comparison (Notebook 07b)
Applied the same decision rule (Recall primary, F1 secondary) across all 4 rows including
the Step 6 unweighted baseline. `LogisticRegression(class_weight='balanced')` decisively won:
Recall=0.6809 (32/47 leavers caught at 0.5 threshold), beating the baseline's 17/47.
RF with `class_weight='balanced'` paradoxically performed worse than the unweighted baseline
on Recall (0.0638) due to bootstrap averaging behaviour.

#### 5. Threshold Selected (Notebook 07b)
The winning model was evaluated at thresholds [0.3, 0.4, 0.5]. Threshold=0.40 was selected
by the user as the production threshold: Recall=0.7872 (37/47), Precision=0.3426, F1=0.4774.
Model saved to `models/attrition_pipeline.joblib`; config written to `models/model_config.json`.

#### 6. SHAP Validation (Notebook 08)
`shap.LinearExplainer` was applied to the production model on the held-out test set.
Top 5 drivers by mean |SHAP value|: OverTime (0.656), YearsSinceLastPromotion (0.566),
TotalWorkingYears (0.557), BusinessTravel_Travel_Frequently (0.518), JobLevel (0.459).
7 of 10 SHAP top features matched Step 6 LR coefficient ranking — confirms LinearExplainer
is working correctly and the model learned interpretable, HR-intuitive signals.
4 of 10 overlapped with XGBoost importances (OverTime, TotalWorkingYears, JobLevel,
YearsWithCurrManager) — these are robust model-agnostic attrition drivers.

---

---
## Step 3 · Create `docs/model_card.md`


In [3]:
today = datetime.now().strftime('%Y-%m-%d')

model_card = f'''# Model Card — Enterprise HR Attrition Risk Model

**Version:** v3  
**Date:** {today}  
**Status:** Production  
**Maintained by:** Enterprise HR AI Project Team  

---

## Model Name

`logreg_balanced_threshold_0.40`  
Algorithm: Logistic Regression (`class_weight='balanced'`, `max_iter=1000`, `solver='lbfgs'`)

---

## Intended Use

**Primary use:** Attrition risk scoring for HR business partners and People Analytics teams.  
The model assigns a probability score (0–1) to each active employee indicating their likelihood
of voluntarily leaving the organization. Employees with a score ≥ 0.40 are flagged for
proactive retention review.

**Intended users:** HR managers, People Analytics teams, department heads conducting retention reviews.  
**Not intended for:** Automated employment decisions, performance management, compensation determination,
or any legally consequential HR action without human review.

---

## Training Data

- **Source:** IBM HR Analytics Employee Attrition & Performance dataset (publicly available on Kaggle).
  This is a **synthetic dataset** created by IBM data scientists for demonstration purposes.
- **Size:** 1,470 employees (1,176 train / 294 test, 80/20 stratified split, `random_state=42`)
- **Features:** 48 engineered features derived from demographics, job characteristics, compensation,
  satisfaction scores, and career history. Encoded with `pd.get_dummies(drop_first=True)` and
  standardized with `StandardScaler` (fitted on training set only).
- **Target:** `Attrition` (binary: 1=Yes/Left, 0=No/Stayed)
- **Class balance:** 83.9% stayed / 16.1% left (84/16 imbalance)

---

## Performance Metrics (Test Set, Threshold = 0.40)

| Metric | Value | Notes |
|:---|:---:|:---|
| **Recall** | **0.7872** | Caught 37 of 47 true leavers (10 missed) |
| **Precision** | **0.3426** | 37 true positives out of 108 flagged employees |
| **F1 Score** | **0.4774** | Harmonic mean of precision and recall |
| **ROC-AUC** | **0.8060** | Ranking quality across all thresholds |
| **Confusion Matrix** | TN=176, FP=71, FN=10, TP=37 | At threshold=0.40 on 294-row test set |

**Decision Threshold:** `0.40` (stored in `models/model_config.json` for use by the API layer;
not hardcoded in application code)

---

## Top 5 SHAP Feature Drivers

Computed using `shap.LinearExplainer` on the held-out test set (Notebook 08).

| Rank | Feature | Mean |SHAP| | HR Plain-Language Meaning |
|:---:|:---|:---:|:---|
| 1 | `OverTime` | 0.6558 | Employees who regularly work overtime are significantly more likely to leave. |
| 2 | `YearsSinceLastPromotion` | 0.5663 | Long stretches without career advancement signal stagnation and flight risk. |
| 3 | `TotalWorkingYears` | 0.5573 | Earlier-career employees have higher mobility; invest in retention from day one. |
| 4 | `BusinessTravel_Travel_Frequently` | 0.5177 | Frequent business travel is a major burnout and attrition driver. |
| 5 | `JobLevel` | 0.4585 | Junior-level employees are more likely to leave; clarify promotion paths. |

---

## Known Limitations

1. **Synthetic training data:** This model was trained on IBM\'s synthetic Kaggle dataset.
   Real-world performance on actual company employee data is **unverified**. Before deploying
   in production, the model should be re-trained or at minimum validated on real organizational data.

2. **Class imbalance ceiling on precision:** The 84/16 class imbalance means that even at
   high recall (0.7872), precision is inherently limited (0.3426 — roughly 1 in 3 flagged
   employees is a true leaver). This is an expected trade-off given the business priority
   of catching leavers (Recall primary). HR teams should be briefed that not every flagged
   employee will leave — flags are risk indicators, not certainties.

3. **JobRole ↔ O*NET taxonomy gap:** The `JobRole` categories in this dataset (e.g.
   \'Laboratory Technician\', \'Sales Representative\') do not map directly to the O*NET
   occupational taxonomy used in the reference dataset (`occupation_master.csv`).
   Role-based skill recommendations planned for Day 3 will require the separate
   `data/external/jobrole_onet_mapping.csv` mapping table, which has not yet been finalized.
   Until that mapping exists, O*NET-derived recommendations cannot be reliably attributed
   to specific job roles.

4. **Static model:** The model captures a point-in-time snapshot of historical attrition patterns.
   It does not update automatically as organizational conditions change. A retraining schedule
   (e.g. quarterly) should be established once deployed on real data.

5. **No causal inference:** SHAP values identify statistical associations, not causal drivers.
   For example, \'OverTime\' being the top driver does not prove that reducing overtime will
   reduce attrition — it means employees who work overtime tend to leave more often.
   Interventions should be designed with HR domain expertise, not derived mechanically from
   model outputs.

---

## Artifact Locations

| Artifact | Path |
|:---|:---|
| Production model | `models/attrition_pipeline.joblib` |
| Model config (threshold) | `models/model_config.json` |
| Feature scaler | `models/scaler.joblib` |
| Model registry | `models/model_registry.json` |
| Archived v1 baseline | `models/baseline_logreg.joblib` |
| Archived v2 candidate | `models/archive/attrition_pipeline_candidate_v1.joblib` |
| SHAP plots | `reports/shap/` (4 PNG files) |

---

## Version History

| Version | Model | Threshold | Recall | F1 | Status |
|:---:|:---|:---:|:---:|:---:|:---|
| v1 | LogReg (unweighted) | 0.50 | 0.3617 | 0.4658 | Superseded |
| v2 | XGBoost (default) | 0.50 | 0.2766 | 0.3714 | Rejected |
| **v3** | **LogReg (balanced)** | **0.40** | **0.7872** | **0.4774** | **Production** |

---

*This model card follows the Model Cards for Model Reporting standard (Mitchell et al., 2019).*
'''

card_path = os.path.join(DOCS, 'model_card.md')
with open(card_path, 'w', encoding='utf-8') as f:
    f.write(model_card)

print(f'Written: {os.path.abspath(card_path)}')
print(f'Size   : {os.path.getsize(card_path):,} bytes')
print(f'Lines  : {model_card.count(chr(10)):,}')

Written: C:\Users\ASUS\Desktop\enterprise_hr_ai\docs\model_card.md
Size   : 5,995 bytes
Lines  : 128


---
## Step 4 · Verification

In [4]:
print('=== DELIVERABLE VERIFICATION ===')
print()

# 1. Registry
reg_path = os.path.join(MODELS, 'model_registry.json')
with open(reg_path) as f:
    reg = json.load(f)
print(f'model_registry.json  : {len(reg["versions"])} versions logged')
for v in reg['versions']:
    print(f"  {v['version']}  {v['name']:<38} status={v['status']:<12} recall={v['metrics']['recall']:.4f}")

print()

# 2. Model card sections
card_path = os.path.join(DOCS, 'model_card.md')
with open(card_path, encoding='utf-8') as f:
    card_text = f.read()

required_sections = [
    ('Model Name',             '## Model Name'),
    ('Intended Use',           '## Intended Use'),
    ('Training Data',          '## Training Data'),
    ('Performance Metrics',    '## Performance Metrics'),
    ('SHAP Feature Drivers',   '## Top 5 SHAP Feature Drivers'),
    ('Known Limitations',      '## Known Limitations'),
    ('Artifact Locations',     '## Artifact Locations'),
    ('Version History',        '## Version History'),
    ('Synthetic data warning',  'synthetic dataset'),
    ('Class imbalance caveat',  '84/16 class imbalance'),
    ('O*NET gap warning',       'O*NET'),
]
print(f'docs/model_card.md   : {os.path.getsize(card_path):,} bytes')
for label, check in required_sections:
    present = check in card_text
    mark = 'OK' if present else 'MISSING'
    print(f'  [{mark}] {label}')

print()
print('=== MODELS DIRECTORY ===')
for fname in sorted(os.listdir(MODELS)):
    fpath = os.path.join(MODELS, fname)
    if os.path.isfile(fpath):
        print(f'  {fname:<40} {os.path.getsize(fpath):>10,} bytes')
    elif os.path.isdir(fpath):
        children = os.listdir(fpath)
        print(f'  {fname}/ ({len(children)} file(s))')
        for c in children:
            cp = os.path.join(fpath, c)
            print(f'    {c:<38} {os.path.getsize(cp):>10,} bytes')

=== DELIVERABLE VERIFICATION ===

model_registry.json  : 3 versions logged
  v1  baseline_logreg                        status=superseded   recall=0.3617
  v2  xgboost_unscaled                       status=rejected     recall=0.2766
  v3  logreg_balanced_threshold_0.40         status=production   recall=0.7872

docs/model_card.md   : 5,995 bytes
  [OK] Model Name
  [OK] Intended Use
  [OK] Training Data
  [OK] Performance Metrics
  [OK] SHAP Feature Drivers
  [OK] Known Limitations
  [OK] Artifact Locations
  [OK] Version History
  [OK] Synthetic data warning
  [OK] Class imbalance caveat
  [OK] O*NET gap warning

=== MODELS DIRECTORY ===
  archive/ (1 file(s))
    attrition_pipeline_candidate_v1.joblib      2,575 bytes
  attrition/ (0 file(s))
  attrition_pipeline.joblib                     2,575 bytes
  baseline_logreg.joblib                        2,575 bytes
  checkpoints/ (0 file(s))
  embeddings/ (0 file(s))
  model_config.json                               195 bytes
  model_regi